# MACHINE LEARNING & ARTIFICIAL INTELLIGENCE FOR DATA SCIENCE - 2TSCPV-2026

## Checkpoint 3 — Detecção de Fraude

### Alex Ribeiro Barros Junior - RM562679

Notebook estruturado para:
1. fazer análise descritiva do problema;
2. levantar hipóteses sobre variáveis relevantes;
3. treinar e comparar ao menos 3 modelos;
4. escolher o melhor modelo com foco em redução de FP e FN.

> Base inspirada no notebook de referência do repositório da FIAP.

## 1) Bibliotecas

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

## 2) Carga dos dados

O notebook de referência usa o arquivo `fraud_study_enrich.csv` do repositório.

In [2]:
DATA_URL = "https://raw.githubusercontent.com/diogenesjusto/FIAP/master/dados/fraud_study_enrich.csv"
df = pd.read_csv(DATA_URL)

print("Formato da base:", df.shape)
display(df.head())
print("Colunas:", list(df.columns))

Formato da base: (20502, 23)


,Unnamed: 0,X,ID_CLIENTE,COD_PARCEIRO_NEGOCIO,COD_NOTA_SERVICO,COD_EMPRESA,DT_MES_EXECUCAO,DESC_MUNICIPIO,DESC_BAIRRO,DESC_CLASSE_CALCULO,...,RESULTADO,SEXO,FaixaIDade,OBITO,ESCOLARIDADE,RENDAESTIMADA,FAIXARENDA,SCORE1,SCORE2,RESULTADO_N
0,1,1,73fe6addf5437873212b29a12c981c15b0db8c06574e61...,700614111,716573134,D002,201412,VOTORANTIM,JD NOVO MUNDO,Residencial,...,IRREGULAR,F,I - Acima de 75 anos,NAO,NaN,380.0,E - DE 0 A 2 SM,10.0,NaN,1
1,2,2,d73cf2fb49fb6538d5c3ade7998b4bacc838b8160e0924...,710914826,715897522,D001,201410,BIRIGUI,CJH JOAO CREVELARO,Residencial,...,REGULAR,F,I - Acima de 75 anos,NAO,NaN,545.0,E - DE 0 A 2 SM,9.0,10.0,0
2,3,3,72afa50efa52af6587f7bbdb486e52b8ecbb91c51c88c9...,701963566,716088410,D001,201411,CAMPINAS,VL PROOST DE SOUZA,Residencial,...,REGULAR,M,E - De 36 a 45 anos,NAO,"COLEGIAL COMPLETO, OU MEDIO COMPLETO",1024.0,E - DE 0 A 2 SM,10.0,8.0,0
3,4,4,a7b351a1166cf63c2023fc81c91e7cb34751f3a188d2cc...,700611368,715792583,D002,201410,SANTOS,CENTRO,Residencial,...,IRREGULAR,I,G - De 56 a 65 anos,NAO,NaN,0.0,E - DE 0 A 2 SM,8.0,10.0,1
4,5,5,bacf56dc47141da2caa3458912a5439551ad65a3d1be5c...,702253670,715719024,D001,201410,SANTO ANTONIO DO ARACANGUA,VICENTINOPOLIS,Residencial,...,REGULAR,M,F - De 46 a 55 anos,NAO,"COLEGIAL COMPLETO, OU MEDIO COMPLETO",9.0,E - DE 0 A 2 SM,9.0,NaN,0


Colunas: ['Unnamed: 0', 'X', 'ID_CLIENTE', 'COD_PARCEIRO_NEGOCIO', 'COD_NOTA_SERVICO', 'COD_EMPRESA', 'DT_MES_EXECUCAO', 'DESC_MUNICIPIO', 'DESC_BAIRRO', 'DESC_CLASSE_CALCULO', 'DT_REFERENCIA', 'NUM_DIAS', 'FATURADO', 'RESULTADO', 'SEXO', 'FaixaIDade', 'OBITO', 'ESCOLARIDADE', 'RENDAESTIMADA', 'FAIXARENDA', 'SCORE1', 'SCORE2', 'RESULTADO_N']


## 3) Análise descritiva

Aqui a ideia é entender:
- volume total de registros;
- proporção de fraude vs. não fraude;
- possíveis variáveis com sinal preditivo;
- existência de valores nulos e duplicados;
- hipóteses de comportamento ao longo do tempo.

In [3]:
target_col = "RESULTADO_N"

print("Resumo geral")
display(df.describe(include="all").T)

print("Valores nulos por coluna")
display(df.isnull().sum().sort_values(ascending=False))

print("Linhas duplicadas:", df.duplicated().sum())

print("Distribuição da variável alvo")
target_counts = df[target_col].value_counts(dropna=False).sort_index()
display(target_counts.to_frame("qtd"))

fraud_rate = df[target_col].mean()
print(f"Taxa de fraude: {fraud_rate:.2%}")

Resumo geral


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,20502.0,NaN,NaN,NaN,24872.335333,14434.995158,1.0,12397.0,24863.5,37353.5,50000.0
X,20502.0,NaN,NaN,NaN,24872.335333,14434.995158,1.0,12397.0,24863.5,37353.5,50000.0
ID_CLIENTE,20502,15214,407773ccec793efc32f4f9f27e8e1a454aca22bc1f4cbf...,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
COD_PARCEIRO_NEGOCIO,20502.0,NaN,NaN,NaN,704064411.185689,20173839.373492,60009716.0,700621219.5,702338717.0,703740444.5,1000032123.0
COD_NOTA_SERVICO,20502.0,NaN,NaN,NaN,715833136.956833,495900.865682,712456387.0,715498598.0,715896543.5,716220786.0,716715961.0
COD_EMPRESA,20502,7,D001,11043,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DT_MES_EXECUCAO,20502.0,NaN,NaN,NaN,201409.787143,1.679057,201407.0,201408.0,201410.0,201411.0,201412.0
DESC_MUNICIPIO,20502,190,RIBEIRAO PRETO,2072,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DESC_BAIRRO,20502,2594,CENTRO,1254,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DESC_CLASSE_CALCULO,20502,5,Residencial,19743,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Valores nulos por coluna


ESCOLARIDADE            11850
SCORE2                   9635
SCORE1                    596
RENDAESTIMADA               2
Unnamed: 0                  0
FATURADO                    0
FAIXARENDA                  0
OBITO                       0
FaixaIDade                  0
SEXO                        0
RESULTADO                   0
NUM_DIAS                    0
X                           0
DT_REFERENCIA               0
DESC_CLASSE_CALCULO         0
DESC_BAIRRO                 0
DESC_MUNICIPIO              0
DT_MES_EXECUCAO             0
COD_EMPRESA                 0
COD_NOTA_SERVICO            0
COD_PARCEIRO_NEGOCIO        0
ID_CLIENTE                  0
RESULTADO_N                 0
dtype: int64

Linhas duplicadas: 0
Distribuição da variável alvo


,qtd
RESULTADO_N,
0,16666
1,3836


Taxa de fraude: 18.71%


In [4]:
# Gráfico simples da distribuição da fraude
ax = df[target_col].value_counts().sort_index().plot(kind="bar")
ax.set_title("Distribuição da classe alvo")
ax.set_xlabel("RESULTADO_N")
ax.set_ylabel("Quantidade")
plt.tight_layout()
plt.show()

### Hipóteses de negócio/padrão

Exemplos de hipóteses:
- fraudes podem se concentrar em certos períodos;
- notas com padrões repetidos ou muito frequentes podem indicar comportamento anômalo;
- a variável de tempo pode melhorar a separação entre fraude e não fraude;
- desbalanceamento de classes pode fazer o modelo errar mais falsos negativos.

## 4) Engenharia de variáveis

Ajuste automático para tentar aproveitar a coluna de data/mês.
Se a coluna `DT_MES_EXECUCAO` vier como data, extraímos ano e mês.
Se vier como numérica ou texto simples, mantemos o valor original e tentamos usar também uma versão tratada.

In [5]:
df_model = df.copy()

if "DT_MES_EXECUCAO" in df_model.columns:
    # DT_MES_EXECUCAO vem como YYYYMM; converter sem format gerava datas em 1970.
    dt = pd.to_datetime(
        df_model["DT_MES_EXECUCAO"].astype("Int64").astype(str),
        format="%Y%m",
        errors="coerce"
    )
    df_model["ANO_EXECUCAO"] = dt.dt.year
    df_model["MES_EXECUCAO"] = dt.dt.month

# Remove colunas totalmente vazias, se houver
df_model = df_model.dropna(axis=1, how="all")

print("Formato apos engenharia:", df_model.shape)
display(df_model.head())

Formato apos engenharia: (20502, 25)


,Unnamed: 0,X,ID_CLIENTE,COD_PARCEIRO_NEGOCIO,COD_NOTA_SERVICO,COD_EMPRESA,DT_MES_EXECUCAO,DESC_MUNICIPIO,DESC_BAIRRO,DESC_CLASSE_CALCULO,...,FaixaIDade,OBITO,ESCOLARIDADE,RENDAESTIMADA,FAIXARENDA,SCORE1,SCORE2,RESULTADO_N,ANO_EXECUCAO,MES_EXECUCAO
0,1,1,73fe6addf5437873212b29a12c981c15b0db8c06574e61...,700614111,716573134,D002,201412,VOTORANTIM,JD NOVO MUNDO,Residencial,...,I - Acima de 75 anos,NAO,NaN,380.0,E - DE 0 A 2 SM,10.0,NaN,1,2014,12
1,2,2,d73cf2fb49fb6538d5c3ade7998b4bacc838b8160e0924...,710914826,715897522,D001,201410,BIRIGUI,CJH JOAO CREVELARO,Residencial,...,I - Acima de 75 anos,NAO,NaN,545.0,E - DE 0 A 2 SM,9.0,10.0,0,2014,10
2,3,3,72afa50efa52af6587f7bbdb486e52b8ecbb91c51c88c9...,701963566,716088410,D001,201411,CAMPINAS,VL PROOST DE SOUZA,Residencial,...,E - De 36 a 45 anos,NAO,"COLEGIAL COMPLETO, OU MEDIO COMPLETO",1024.0,E - DE 0 A 2 SM,10.0,8.0,0,2014,11
3,4,4,a7b351a1166cf63c2023fc81c91e7cb34751f3a188d2cc...,700611368,715792583,D002,201410,SANTOS,CENTRO,Residencial,...,G - De 56 a 65 anos,NAO,NaN,0.0,E - DE 0 A 2 SM,8.0,10.0,1,2014,10
4,5,5,bacf56dc47141da2caa3458912a5439551ad65a3d1be5c...,702253670,715719024,D001,201410,SANTO ANTONIO DO ARACANGUA,VICENTINOPOLIS,Residencial,...,F - De 46 a 55 anos,NAO,"COLEGIAL COMPLETO, OU MEDIO COMPLETO",9.0,E - DE 0 A 2 SM,9.0,NaN,0,2014,10


## 5) Separação entre treino e teste

Usamos `stratify` para preservar a proporção da fraude nas duas amostras.

In [6]:
leakage_cols = [
    target_col,
    "RESULTADO",  # versao textual do alvo; manter esta coluna vaza a resposta
]

id_cols = [
    "Unnamed: 0",
    "X",
    "ID_CLIENTE",
    "COD_PARCEIRO_NEGOCIO",
    "COD_NOTA_SERVICO",
]

cols_to_drop = [c for c in leakage_cols + id_cols if c in df_model.columns]

X = df_model.drop(columns=cols_to_drop)
y = df_model[target_col].astype(int)

print("Colunas removidas:", cols_to_drop)
print("Total de features usadas:", X.shape[1])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste :", X_test.shape)
print("Fraude treino:", y_train.mean())
print("Fraude teste :", y_test.mean())

## 6) Pré-processamento

Separação automática entre colunas numéricas e categóricas.

In [7]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Numéricas:", numeric_features)
print("Categóricas:", categorical_features)

## 7) Funções de avaliação

A métrica principal aqui não é só accuracy.  
Fraude costuma ser um problema desbalanceado, então FP e FN precisam ser vistos separadamente.

In [8]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = None

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "modelo": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "TP": tp
    }

    if y_score is not None:
        metrics["roc_auc"] = roc_auc_score(y_test, y_score)
    else:
        metrics["roc_auc"] = np.nan

    return metrics, cm

def print_cm(cm, title):
    cm_df = pd.DataFrame(cm, index=["Real 0", "Real 1"], columns=["Prev 0", "Prev 1"])
    print(title)
    display(cm_df)

## 8) Modelo 1 — baseline do enunciado

DecisionTree simples, no estilo do notebook de exemplo.

In [9]:
model_1 = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", DecisionTreeClassifier(random_state=42))
])

model_1.fit(X_train, y_train)
m1_metrics, m1_cm = evaluate_model("DecisionTree baseline", model_1, X_test, y_test)
print_cm(m1_cm, "Matriz de confusão — DecisionTree baseline")
print(classification_report(y_test, model_1.predict(X_test), zero_division=0))

Matriz de confusão — DecisionTree baseline


,Prev 0,Prev 1
Real 0,4560,440
Real 1,575,576


              precision    recall  f1-score   support

           0       0.89      0.91      0.90      5000
           1       0.57      0.50      0.53      1151

    accuracy                           0.83      6151
   macro avg       0.73      0.71      0.72      6151
weighted avg       0.83      0.83      0.83      6151



## 9) Modelo 2 — DecisionTree com ajuste para desbalanceamento

Aqui a intenção é reduzir erros da classe fraudulenta, que normalmente é a classe minoritária.

In [10]:
model_2 = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced",
        max_depth=6,
        min_samples_leaf=20,
        min_samples_split=40
    ))
])

model_2.fit(X_train, y_train)
m2_metrics, m2_cm = evaluate_model("DecisionTree balanceado", model_2, X_test, y_test)
print_cm(m2_cm, "Matriz de confusão — DecisionTree balanceado")
print(classification_report(y_test, model_2.predict(X_test), zero_division=0))

Matriz de confusão — DecisionTree balanceado


,Prev 0,Prev 1
Real 0,4594,406
Real 1,787,364


              precision    recall  f1-score   support

           0       0.85      0.92      0.89      5000
           1       0.47      0.32      0.38      1151

    accuracy                           0.81      6151
   macro avg       0.66      0.62      0.63      6151
weighted avg       0.78      0.81      0.79      6151



## 10) Modelo 3 — Random Forest balanceado

Modelo mais robusto que a árvore simples e geralmente com melhor generalização.

In [11]:
model_3 = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced_subsample",
        max_depth=None,
        min_samples_leaf=5,
        n_jobs=-1
    ))
])

model_3.fit(X_train, y_train)
m3_metrics, m3_cm = evaluate_model("RandomForest balanceado", model_3, X_test, y_test)
print_cm(m3_cm, "Matriz de confusão — RandomForest balanceado")
print(classification_report(y_test, model_3.predict(X_test), zero_division=0))

Matriz de confusão — RandomForest balanceado


,Prev 0,Prev 1
Real 0,4120,880
Real 1,560,591


              precision    recall  f1-score   support

           0       0.88      0.82      0.85      5000
           1       0.40      0.51      0.45      1151

    accuracy                           0.77      6151
   macro avg       0.64      0.67      0.65      6151
weighted avg       0.79      0.77      0.78      6151



## 11) Tabela comparativa

A leitura final deve olhar principalmente para:
- **FN**: fraude não detectada;
- **FP**: transação normal sinalizada como fraude;
- **Recall**: quanto da fraude foi capturada;
- **ROC AUC** e **balanced accuracy**: qualidade geral em base desbalanceada.

In [12]:
results = pd.DataFrame([m1_metrics, m2_metrics, m3_metrics])
results = results[[
    "modelo", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "FP", "FN", "TN", "TP"
]].sort_values(by=["FN", "recall", "roc_auc", "FP"], ascending=[True, False, False, True])

display(results)

## 12) Escolha do melhor modelo

Critério sugerido para a entrega:
- escolher o modelo com menor quantidade de **FN**;
- em caso de empate, priorizar maior **recall** e maior **ROC AUC**;
- olhar **FP** como custo operacional de investigação;
- justificar com base no impacto do negócio.

In [ ]:
best_model_name = results.iloc[0]["modelo"]
print("Modelo com melhor desempenho segundo a tabela:", best_model_name)

# O RandomForest detecta mais fraudes, com menos FN, mas gera mais falsos positivos. 
# Já a DecisionTree baseline tem melhor F1 e menos FP, mas deixa passar mais fraudes. 
# Para fraude, faz sentido priorizar FN, mas vale explicar esse trade-off na entrega.

## Considerações finais

Neste projeto, meu objetivo foi construir e avaliar modelos de classificação para identificar possíveis fraudes, usando a variável **RESULTADO_N** como alvo. A partir da base disponível, busquei entender os dados, analisar a distribuição entre registros regulares e irregulares e preparar as variáveis para que os modelos conseguissem aprender padrões relevantes.

Ao longo do notebook, organizei o processo em etapas: carregamento da base, análise descritiva, tratamento de valores nulos, engenharia de variáveis, separação entre treino e teste, pré-processamento e comparação dos modelos. Também utilizei pipelines para deixar o fluxo mais estruturado, aplicando imputação, padronização das variáveis numéricas e codificação das variáveis categóricas.

Testei três abordagens: uma árvore de decisão simples, uma árvore de decisão com ajuste para desbalanceamento e um modelo Random Forest balanceado. Como se trata de um problema de fraude, não considerei apenas a acurácia; dei mais atenção a métricas como recall, falsos negativos, falsos positivos, F1, balanced accuracy e ROC AUC, pois elas mostram melhor o impacto do modelo na prática.

Durante a revisão, percebi a importância de evitar vazamento de dados, principalmente removendo variáveis que poderiam entregar a resposta diretamente ao modelo. Com os ajustes, os resultados ficaram mais realistas. O Random Forest teve melhor desempenho para capturar fraudes e reduzir falsos negativos, embora tenha aumentado os falsos positivos. Por isso, entendo que a escolha do modelo deve considerar o equilíbrio entre não deixar fraudes passarem e o custo de investigar alertas incorretos.